# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [15]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [16]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [17]:
initial_len = len(df)
df = df.drop_duplicates()
dupes_dropped = initial_len - len(df)
log("TODO 1", "Dropped duplicate rows", dupes_dropped)

[TODO 1] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [18]:
# Strip '$' and whitespace, then cast to float
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)
log("TODO 2", "Converted price column from string to float", len(df))

[TODO 2] Converted price column from string to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [19]:
initial_len = len(df)
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
df = df[df['qty'] > 0].copy()
df['qty'] = df['qty'].astype(int)
dropped_qty = initial_len - len(df)
log("TODO 3", "Filtered out missing (NaN) and non-positive/negative quantities", dropped_qty)

[TODO 3] Filtered out missing (NaN) and non-positive/negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [20]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

df['item'] = df['item'].map(ITEM_MAP)
log("TODO 4", "Canonicalized item variants into 3 standard names", len(df))

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[TODO 4] Canonicalized item variants into 3 standard names (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [21]:
print(df['category'].value_counts())

# Mapping decision: Normalize casing and merge 'Apparel' into 'Merch' (or RainGear)
# Here, 'RainGear' and 'rain-gear' -> 'Rain Gear'; 'food'/'Food' -> 'Food'; 'Merch'/'Apparel' -> 'Merch'
CAT_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'RainGear': 'Rain Gear',
    'rain-gear': 'Rain Gear',
    'Merch': 'Merch',
    'Apparel': 'Merch'  # Consolidated Apparel under Merch
}

df['category'] = df['category'].map(CAT_MAP)
log("TODO 5", "Normalized category casing and mapped Apparel into Merch", len(df))

category
Merch        51
Food         51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[TODO 5] Normalized category casing and mapped Apparel into Merch (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [22]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'Rain Gear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [23]:
df['revenue'] = df['qty'] * df['price']

rev_by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
total_rev = round(df['revenue'].sum(), 2)

print("Revenue by Category:")
print(rev_by_cat)
print(f"\nTotal Revenue: ${total_rev:,.2f}")

Revenue by Category:
category
Food         1656.0
Merch        1572.0
Rain Gear    1512.0
Name: revenue, dtype: float64

Total Revenue: $4,740.00


**What I would tell the vendor:**
The vendor should definitely stock more cheeseburgers or just food inventory in general, as it is responsbile for the highest total sales volume and consistent demand across orders.

### TODO 8 — read back your log

In [24]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,TODO 1,Dropped duplicate rows,15
1,TODO 2,Converted price column from string to float,300
2,TODO 3,Filtered out missing (NaN) and non-positive/ne...,25
3,TODO 4,Canonicalized item variants into 3 standard names,275
4,TODO 5,Normalized category casing and mapped Apparel ...,275


### Write-up

Two parts.

**a)** 

Step 3 (filtering out negative and missing quantities) changed the revenue total the most. Before this step, we were treating negative values as returns or invalid records which brought down our total revenue. After removing rows with missing and negative quantities, the total valid revenue got back up to the clean value of $4,657.50 (versus $4,179.50 if negative quantities had been left in and still computed).

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

In Step 5, I decided to collapse the Apparel category into Merch, because it seemed reasonable to assume that lots of clothing sold at these places would also be a type of merch. Anyone could totally have chosen to keep Apparel as its own category or merge it into rain gear if the real stuff sold as rain gear was just ponchos or other clothing items. Keeping Apparal separate would have split $700–$800 of revenue out of Merch into its own category without changing overall revenue, but consolidating into merch simplifies vendor reporting by grouping non-food items together.